# Modelagem e Avaliação

Este notebook implementa as melhorias solicitadas pelo professor:

1. **Baseline Naive**: Referência mínima para comparação
2. **Predição Recursiva**: Multi-step forecasting
3. **MultiOutputRegressor (MIMO)**: Previsão de 7 dias simultaneamente
4. **TimeSeriesSplit**: Validação cruzada temporal no GridSearchCV
5. **Experimentos com Janelas**: Comparação de W=3, 7, 15

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import joblib

sys.path.append(os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'src'))
from modeling import (
    calcular_metricas, baseline_naive, treinar_modelo,
    previsao_recursiva, treinar_multioutput, despadronizar
)

pd.options.display.float_format = '{:.2f}'.format
sns.set_style('whitegrid')

## 1. Carregamento dos Dados Processados

In [ ]:
df_train = pd.read_parquet("../data/processed/train_features.parquet", engine="pyarrow")
df_test = pd.read_parquet("../data/processed/test_features.parquet", engine="pyarrow")

print(f"Treino: {df_train.shape}")
print(f"Teste: {df_test.shape}")

In [ ]:
df_train.head()

In [ ]:
y_train = df_train["Quantidade"]
X_train = df_train.drop(columns=["Data", "Quantidade"])

y_test = df_test["Quantidade"]
X_test = df_test.drop(columns=["Data", "Quantidade"])

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

In [ ]:
scaler = joblib.load('../models/scaler_quantidade.pkl')
media_real = scaler.mean_[0]
escala_real = scaler.scale_[0]

print(f"Média: {media_real:.2f}, Desvio Padrão: {escala_real:.2f}")

## 2. Baseline Naive (Persistence Model)

**Propósito**: Estabelecer referência mínima. Se modelos complexos não superam o baseline, há problema na abordagem.

In [ ]:
y_test_real, y_pred_baseline = baseline_naive(y_train, y_test)

y_test_real_units = despadronizar(y_test_real, media_real, escala_real)
y_pred_baseline_units = despadronizar(y_pred_baseline, media_real, escala_real)

metricas_baseline = calcular_metricas(y_test_real_units, y_pred_baseline_units)
print("Baseline Naive:")
print(f"  MAE: {metricas_baseline['MAE']:.2f}")
print(f"  RMSE: {metricas_baseline['RMSE']:.2f}")
print(f"  MAPE: {metricas_baseline['MAPE']:.2f}%")

## 3. Treinamento dos Modelos

### Modelos Base com GridSearchCV e TimeSeriesSplit

In [ ]:
modelo_lr, metricas_lr, y_pred_lr = treinar_modelo(
    'lr', X_train, y_train, X_test, y_test, usar_gridsearch=False
)

y_test_units = despadronizar(y_test, media_real, escala_real)
y_pred_lr_units = despadronizar(y_pred_lr, media_real, escala_real)
metricas_lr_real = calcular_metricas(y_test_units, y_pred_lr_units)

print("Linear Regression:")
print(f"  MAE: {metricas_lr_real['MAE']:.2f}")
print(f"  RMSE: {metricas_lr_real['RMSE']:.2f}")
print(f"  MAPE: {metricas_lr_real['MAPE']:.2f}%")

In [ ]:
modelo_rf, metricas_rf, y_pred_rf = treinar_modelo(
    'rf', X_train, y_train, X_test, y_test, usar_gridsearch=True
)

y_pred_rf_units = despadronizar(y_pred_rf, media_real, escala_real)
metricas_rf_real = calcular_metricas(y_test_units, y_pred_rf_units)

print("\nRandom Forest (Otimizado com TimeSeriesSplit):")
print(f"  MAE: {metricas_rf_real['MAE']:.2f}")
print(f"  RMSE: {metricas_rf_real['RMSE']:.2f}")
print(f"  MAPE: {metricas_rf_real['MAPE']:.2f}%")

In [ ]:
modelo_gb, metricas_gb, y_pred_gb = treinar_modelo(
    'gb', X_train, y_train, X_test, y_test, usar_gridsearch=False
)

y_pred_gb_units = despadronizar(y_pred_gb, media_real, escala_real)
metricas_gb_real = calcular_metricas(y_test_units, y_pred_gb_units)

print("\nGradient Boosting:")
print(f"  MAE: {metricas_gb_real['MAE']:.2f}")
print(f"  RMSE: {metricas_gb_real['RMSE']:.2f}")
print(f"  MAPE: {metricas_gb_real['MAPE']:.2f}%")

In [ ]:
modelo_et, metricas_et, y_pred_et = treinar_modelo(
    'et', X_train, y_train, X_test, y_test, usar_gridsearch=False
)

y_pred_et_units = despadronizar(y_pred_et, media_real, escala_real)
metricas_et_real = calcular_metricas(y_test_units, y_pred_et_units)

print("\nExtra Trees:")
print(f"  MAE: {metricas_et_real['MAE']:.2f}")
print(f"  RMSE: {metricas_et_real['RMSE']:.2f}")
print(f"  MAPE: {metricas_et_real['MAPE']:.2f}%")

## 4. Comparação de Modelos (One-Step Ahead)

In [ ]:
resultados = [
    {'Modelo': 'Baseline Naive', 'MAE': metricas_baseline['MAE'], 'RMSE': metricas_baseline['RMSE'], 'MAPE': metricas_baseline['MAPE']},
    {'Modelo': 'Linear Regression', 'MAE': metricas_lr_real['MAE'], 'RMSE': metricas_lr_real['RMSE'], 'MAPE': metricas_lr_real['MAPE']},
    {'Modelo': 'Random Forest', 'MAE': metricas_rf_real['MAE'], 'RMSE': metricas_rf_real['RMSE'], 'MAPE': metricas_rf_real['MAPE']},
    {'Modelo': 'Gradient Boosting', 'MAE': metricas_gb_real['MAE'], 'RMSE': metricas_gb_real['RMSE'], 'MAPE': metricas_gb_real['MAPE']},
    {'Modelo': 'Extra Trees', 'MAE': metricas_et_real['MAE'], 'RMSE': metricas_et_real['RMSE'], 'MAPE': metricas_et_real['MAPE']}
]

df_comparacao = pd.DataFrame(resultados).sort_values('MAPE')
print("Comparação de Modelos (One-Step Ahead):")
df_comparacao

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_test_units.values, label='Demanda Real', color='blue', alpha=0.6)
plt.plot(y_pred_et_units, label='Extra Trees (Melhor)', color='red', linestyle='--')
plt.plot(y_pred_baseline_units.values, label='Baseline Naive', color='gray', linestyle=':', alpha=0.7)
plt.title('Comparação: Realidade vs Modelos (Período de Teste)')
plt.xlabel('Dias')
plt.ylabel('Quantidade')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Predição Recursiva Multi-Step

**Abordagem**: A previsão do dia t alimenta a previsão do dia t+1.

**Desafio**: Erro acumula ao longo do horizonte.

In [ ]:
X_ultimo_dia = X_test.iloc[[-1]]

previsoes_recursivas = previsao_recursiva(modelo_et, X_ultimo_dia, steps=7)

previsoes_recursivas_units = despadronizar(previsoes_recursivas, media_real, escala_real)

print("Previsões Recursivas (7 dias à frente):")
for i, pred in enumerate(previsoes_recursivas_units, 1):
    print(f"  Dia +{i}: {pred:.2f} unidades")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, 8), previsoes_recursivas_units, marker='o', linestyle='-', color='red', label='Previsão Recursiva')
plt.title('Predição Recursiva: 7 Dias à Frente (Extra Trees)')
plt.xlabel('Horizonte (dias)')
plt.ylabel('Quantidade Prevista')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. MultiOutputRegressor (MIMO)

**Abordagem**: Prever 7 dias simultaneamente em um único modelo.

In [ ]:
df_mimo = pd.read_parquet("../data/processed/mimo_features.parquet", engine="pyarrow")
print(f"Shape MIMO: {df_mimo.shape}")
df_mimo.head()

In [ ]:
target_cols = [f'target_d{i}' for i in range(1, 8)]

X_mimo = df_mimo.drop(columns=['Data', 'Quantidade'] + target_cols)
y_mimo = df_mimo[target_cols]

split_mimo = int(len(df_mimo) * 0.8)
X_mimo_train = X_mimo.iloc[:split_mimo]
y_mimo_train = y_mimo.iloc[:split_mimo]
X_mimo_test = X_mimo.iloc[split_mimo:]
y_mimo_test = y_mimo.iloc[split_mimo:]

print(f"X_mimo_train: {X_mimo_train.shape}, y_mimo_train: {y_mimo_train.shape}")

In [ ]:
modelo_mimo, metricas_mimo, y_pred_mimo = treinar_multioutput(
    X_mimo_train, y_mimo_train, X_mimo_test, horizonte=7
)

df_metricas_mimo = pd.DataFrame(metricas_mimo)
print("Métricas MIMO por Horizonte:")
df_metricas_mimo

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_metricas_mimo['horizonte'], df_metricas_mimo['MAPE'], marker='o', linestyle='-', color='green')
plt.title('Erro (MAPE) por Horizonte de Previsão (MIMO)')
plt.xlabel('Horizonte')
plt.ylabel('MAPE (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importancias = modelo_et.feature_importances_
nomes_colunas = X_train.columns

df_importancia = pd.DataFrame({'Atributo': nomes_colunas, 'Importancia': importancias})
df_importancia = df_importancia.sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x='Importancia', y='Atributo', data=df_importancia, palette='magma')
plt.title('Importância das Variáveis (Extra Trees)')
plt.xlabel('Pontuação de Importância')
plt.ylabel('Atributos')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 8. Experimentos com Diferentes Janelas

Comparação de performance com diferentes configurações de lags e janelas rolling.

In [ ]:
configs_nomes = ['lag3_w3', 'lag7_w7', 'lag15_w15', 'lag7_w3', 'lag7_w15', 'lag3_w7']
resultados_janelas = []

for nome in configs_nomes:
    df_t = pd.read_parquet(f"../data/processed/train_{nome}.parquet", engine="pyarrow")
    df_te = pd.read_parquet(f"../data/processed/test_{nome}.parquet", engine="pyarrow")
    
    y_t = df_t["Quantidade"]
    X_t = df_t.drop(columns=["Data", "Quantidade"])
    y_te = df_te["Quantidade"]
    X_te = df_te.drop(columns=["Data", "Quantidade"])
    
    _, _, y_pred = treinar_modelo('et', X_t, y_t, X_te, y_te, usar_gridsearch=False)
    
    scaler_temp = joblib.load('../models/scaler_quantidade.pkl')
    y_te_units = despadronizar(y_te, scaler_temp.mean_[0], scaler_temp.scale_[0])
    y_pred_units = despadronizar(y_pred, scaler_temp.mean_[0], scaler_temp.scale_[0])
    
    metricas = calcular_metricas(y_te_units, y_pred_units)
    resultados_janelas.append({
        'Configuração': nome,
        'MAE': metricas['MAE'],
        'RMSE': metricas['RMSE'],
        'MAPE': metricas['MAPE']
    })

df_janelas = pd.DataFrame(resultados_janelas).sort_values('MAPE')
print("Comparação de Configurações de Janelas:")
df_janelas

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_janelas['Configuração'], df_janelas['MAPE'], marker='o', linestyle='-', color='purple')
plt.title('Performance (MAPE) por Configuração de Janela')
plt.xlabel('Configuração (lags_janela)')
plt.ylabel('MAPE (%)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Conclusões

### Resumo das Melhorias Implementadas

1. ✓ **Baseline Naive**: Estabelecida referência mínima
2. ✓ **TimeSeriesSplit**: Validação cruzada temporal no GridSearchCV
3. ✓ **Predição Recursiva**: Implementada abordagem multi-step
4. ✓ **MIMO**: MultiOutputRegressor para previsão de 7 dias
5. ✓ **Experimentos com Janelas**: Comparação de W=3, 7, 15

### Próximos Passos

- Análise detalhada dos resultados
- Comparação recursivo vs MIMO
- Documentação para o TCC
- Visualizações finais